# Heat conduction topology optimization

Reproduction of the 91-line MATLAB heat-conduction example in Section 5.1.6
of Bendsøe and Sigmund (2004). The reference call is
`toph(40,40,0.4,3.0,1.2)`, with reference objective `447.9944`.

> Bendsøe, M. P., and Sigmund, O. *Topology Optimization: Theory, Methods,
> and Applications*. Springer, 2004.

In [ ]:
import jax
import jax.numpy as np

jax.config.update("jax_enable_x64", True)

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as onp
from IPython.display import Image as DisplayImage
from IPython.display import display
from jax_fem import logger
from PIL import Image as PILImage

logger.setLevel("WARNING")

from topax.filter import build_conv_filter

In [ ]:
"""

2D heat conduction problem

Minimize thermal compliance under a material volume constraint.

"""


from jax_fem.generate_mesh import Mesh, get_meshio_cell_type, rectangle_mesh
from jax_fem.solver import ad_wrapper

from topax.problem import TopOptProblem


class HeatConduction(TopOptProblem):

    def custom_init(self):
        self.fe = self.fes[0]
        self.fe.flex_inds = np.arange(len(self.fe.cells))
        self.nodal_heat = 0.01 * np.ones((self.fe.num_total_nodes, 1))
        self.add_nodal_load(self.nodal_heat)

    def get_tensor_map(self):
        def heat_flux(T_grad, xPhys):
            conductivity = 0.001 + 0.999 * xPhys**3
            return conductivity * T_grad
        return heat_flux

    def set_params(self, params):
        full_params = np.ones((self.fe.num_cells, params.shape[1]))
        full_params = full_params.at[self.fe.flex_inds].set(params)
        thetas = np.repeat(full_params[:, None, :], self.fe.num_quads, axis=1)
        self.full_params = full_params
        self.internal_vars = [thetas]


def prep_fem(Nx, Ny, Lx, Ly):

    ele_type = 'QUAD4'
    cell_type = get_meshio_cell_type(ele_type)
    meshio_mesh = rectangle_mesh(Nx, Ny, domain_x=Lx, domain_y=Ly)
    mesh = Mesh(meshio_mesh.points, meshio_mesh.cells_dict[cell_type], ele_type)

    def heat_sink(point):
        return np.logical_and(
            np.isclose(point[0], 0., atol=1e-5),
            np.isclose(point[1], Ly / 2., atol=Ly / 20. + 1e-5),
        )

    problem = HeatConduction(
        mesh,
        vec=1,
        dim=2,
        ele_type=ele_type,
        dirichlet_bc_info=[[heat_sink], [0], [lambda point: 0.]],
    )

    solver_options = {'petsc_solver': {'ksp_type': 'preonly', 'pc_type': 'lu'}}
    fwd_pred = ad_wrapper(
        problem,
        solver_options=solver_options,
        adjoint_solver_options=solver_options,
    )

    return fwd_pred, problem

In [ ]:
# SETUP
vf = 0.4
rmin = 1.2

Nx, Ny = 40, 40
Lx, Ly = Nx, Ny
fwd_pred, problem = prep_fem(Nx, Ny, Lx, Ly)


def J_total(xPhys):
    T = fwd_pred(xPhys)[0]
    return np.sum(problem.nodal_heat * T)


def volume_constraint(xPhys):
    return np.sum(xPhys) - vf * xPhys.size


H, Hs = build_conv_filter(problem, rmin=rmin)

def sensitivity_filter(dc, x):
    dc_col = dc.reshape(-1, 1)
    x_col = x.reshape(-1, 1)
    numerator = H @ (dc_col * x_col)
    denominator = Hs * np.maximum(x_col, 1e-3)
    return (numerator / denominator).reshape(dc.shape)


def oc_update(x, dc):
    l1 = 0.0
    l2 = 1e5
    move = 0.2
    while l2 - l1 > 1e-4:
        lmid = 0.5 * (l2 + l1)
        xnew = np.maximum(
            0.001,
            np.maximum(
                x - move,
                np.minimum(1.0, np.minimum(x + move, x * np.sqrt(-dc / lmid))),
            ),
        )
        if np.sum(xnew) > vf * Nx * Ny:
            l1 = lmid
        else:
            l2 = lmid
    return xnew

x0 = vf * np.ones((Nx * Ny, 1))

In [ ]:
# OPTIMIZATION LOOP
loop = 0
change = 1
xnew = x0
frames = []
while change > 0.01:
    loop += 1
    J, dJ = jax.value_and_grad(J_total)(xnew)
    xold = xnew.copy()
    dJ = sensitivity_filter(dJ, xold)
    xnew = oc_update(xold, dJ)
    xPhys = xnew
    vol = np.mean(xPhys)
    change = np.max(np.abs(xnew - xold))
    print(f' It.:{loop:5d}, Obj.:{J:11.4f}, Vol.:{vol:7.3f}, ch.:{change:7.3f}')
    field = onp.flip(xPhys.reshape(Ny, Nx, order='F'), axis=0)
    frames.append(onp.asarray(field))

In [ ]:
# SAVE OPTIMIZATION HISTORY
output_path = Path("docs/imgs/example_topopt_heat.gif")
output_path.parent.mkdir(parents=True, exist_ok=True)

gif_frames = []
for field in frames:
    rgba = plt.get_cmap("gray_r")(onp.clip(field, 0.0, 1.0), bytes=True)
    image = PILImage.fromarray(rgba)
    image = image.resize((Nx * 8, Ny * 8), PILImage.Resampling.NEAREST)
    gif_frames.append(image)

gif_frames[0].save(
    output_path,
    save_all=True,
    append_images=gif_frames[1:],
    duration=100,
    loop=0,
)
display(DisplayImage(filename=str(output_path)))